# Discriminative performance - FPR and Recall

Reproduces **Tables 4-5 (FPR)** and **Figures 6-7 (Recall)**.

- **FPR** is measured on *negative* (non-plagiarised) pairs - visually different
  images from the same domain that share no valid transformation. A pair is a
  false positive when the model predicts a non-empty sequence (score >= threshold).
- **Recall** is measured on *positive* pairs - an image and an augmented copy of
  itself; recall is the fraction correctly flagged as plagiarised.

Each model is wrapped in a `sim_fn(img1, img2) -> score`. For our models the
score is `1` unless the decoder emits the empty sequence on the first step (the
single-step rule of Section 4.4), giving a binary {0,1} score.

- **DomainNet** (public): `recall_fpr_bench.DomainFprEvaluator` /
  `DomainRecallEvaluator`.
- **Curated negatives** (NOT public; external repo): `FprEvaluator` /
  `RecallRobustnessEvaluator` from the authors' dataset-generation project
  (`pairwise_comparison_validation`). Point `DATASET_GEN_SRC` at that repo.


In [ ]:
# ============================================================
# Paths - edit these to point at YOUR local files.
# Nothing below is tied to a specific machine.
# ============================================================
import os

# Model configs shipped with the repo:
CONFIG_EFFNET = "../configs/train_config_effnet.yaml"
CONFIG_VIT    = "../configs/train_config_vit.yaml"

# Model weights: point these at the checkpoints YOU trained.
#   *_PRE : after self-supervised pre-training -> scripts/run_train.py  / run_train_siamnet.py
#   *_SFT : after supervised fine-tuning       -> scripts/run_tune.py   / run_tune_siamnet.py
# Each file is a training checkpoint containing a "model_state_dict" entry.
EFFNET_PRE  = "PATH/TO/effnet_pretrain/checkpoint_epoch_XX.pth"
EFFNET_SFT  = "PATH/TO/effnet_tune/checkpoint_epoch_XX.pth"
VIT_PRE     = "PATH/TO/vit_pretrain/checkpoint_epoch_XX.pth"
VIT_SFT     = "PATH/TO/vit_tune/checkpoint_epoch_XX.pth"
SIAMNET_PRE = "PATH/TO/siamnet_pretrain/checkpoint_epoch_XX.pth"
SIAMNET_SFT = "PATH/TO/siamnet_tune/checkpoint_epoch_XX.pth"

# Datasets:
DOMAINNET_ROOT = "PATH/TO/domainnet"       # public: http://ai.bu.edu/M3SDA/
NEGATIVE_ROOT  = "PATH/TO/negative_pairs"  # curated negatives - NOT public (see paper, Data availability)
NEGATIVE_JSON  = "filtered_negative_test_dataset_meta.json"  # produced by filter_good_pairs.py

# External evaluators / dataset for the curated negative set live in the authors'
# dataset-generation repo (pairwise_comparison_validation, categorization_visualization).
DATASET_GEN_SRC = "PATH/TO/dataset-generation/src"

# Where benchmark JSON / figures are written:
OUTPUT_DIR = "metrics"


In [ ]:
import os, sys, io, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
from torchvision import transforms

# Make the repo importable (this notebook lives in benchmarking/).
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.dataset import (
    ImageTransformer, TransformTokenizer,
    DomainNetDataset, NegativeImagePairDataset, SimpleDomainNetDataset,
)
from src.model import ImageTransformPredictor, SiamNet

%load_ext autoreload
%autoreload 2


In [ ]:
# The curated-negative-set evaluators live in the authors' dataset-generation
# repo (the curated dataset itself is not public - see the paper, Data availability).
if DATASET_GEN_SRC not in sys.path:
    sys.path.insert(0, DATASET_GEN_SRC)

from pairwise_comparison_validation import (
    RecallRobustnessEvaluator, plot_recall_robustness, FprEvaluator, plot_fpr_robustness,
)
from categorization_visualization import plot_classification_histogram, DatasetSampler


### DomainNet Bench

In [ ]:
from recall_fpr_bench import DomainRecallEvaluator, DomainFprEvaluator
from src.dataset import SimpleDomainNetDataset

In [ ]:
tokenizer = TransformTokenizer()
transformer = ImageTransformer()

In [ ]:
path = DOMAINNET_ROOT
data =  SimpleDomainNetDataset(path, split="val")

In [ ]:
data.domain_distribution

In [ ]:
image, image_class = data[17258]
image

In [ ]:
preprocess = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

config = OmegaConf.load(CONFIG_VIT)
model = ImageTransformPredictor(config.model)

checkpoint_path = VIT_PRE
# checkpoint_path = VIT_SFT

checkpoint = torch.load(checkpoint_path, map_location='cuda', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to('cuda')
model.eval
print('ready')

def sim_fn_itp_vit(img1, img2):
    image_tensor_1 = preprocess(img1).unsqueeze(0)
    image_tensor_2 = preprocess(img2).unsqueeze(0)

    image_tensor_1 = image_tensor_1.to(next(model.parameters()).device)
    image_tensor_2 = image_tensor_2.to(next(model.parameters()).device)
        
    generated = model.generate(image_tensor_1, image_tensor_2, max_new_tokens=10, do_sample=False)
    if generated[0][1] == 2: # выдало пустую последовательность
        similarity_score = 0.
    else:
        similarity_score = 1.

    del image_tensor_1
    del image_tensor_2

    return similarity_score

In [ ]:
preprocess = transforms.Compose([
            transforms.Resize((300, 300)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

config = OmegaConf.load(CONFIG_EFFNET)
model = ImageTransformPredictor(config.model)

checkpoint_path = EFFNET_PRE
# checkpoint_path = EFFNET_SFT

checkpoint = torch.load(checkpoint_path, map_location='cuda', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to('cuda')
model.eval()
print('ready')

# model.load_state_dict(torch.load('checkpoints/itp_011/model_weights.pth'))
# model.to('cpu')
# model.eval()
# print('ready')

def sim_fn_itp_effnet(img1, img2):
    image_tensor_1 = preprocess(img1).unsqueeze(0)
    image_tensor_2 = preprocess(img2).unsqueeze(0)

    image_tensor_1 = image_tensor_1.to(next(model.parameters()).device)
    image_tensor_2 = image_tensor_2.to(next(model.parameters()).device)
    
    generated = model.generate(image_tensor_1, image_tensor_2, max_new_tokens=10, do_sample=False)
    if generated[0][1] == 2: # выдало пустую последовательность
        similarity_score = 0.
    else:
        similarity_score = 1.
    del image_tensor_1
    del image_tensor_2

    return similarity_score

In [ ]:
import re
import torch

def sim_fn_qwen(img1, img2):
    was_training = model.training
    model.eval()
    
    try:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": img1},
                {"type": "image", "image": img2},
                {"type": "text", "text": prompt_class_prediction},
            ],
        }]

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.inference_mode():  # ← строже, чем no_grad
            generated_ids = model.generate(**inputs, max_new_tokens=128)

        # Извлечение текста
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0].strip()

        # === КРИТИЧЕСКАЯ ОЧИСТКА ===
        del generated_ids_trimmed
        del generated_ids
        del inputs
        del image_inputs
        del video_inputs
        del messages
        del text

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        match = re.fullmatch(r'[01]', output_text)
        if match:
            return float(match.group(0))
        else:
            raise ValueError(f"Unexpected model output: '{output_text}'. Expected '0' or '1'.")

    finally:
        if was_training:
            model.train()

In [ ]:
model = SiamNet()

preprocess = transforms.Compose([
            transforms.Resize((300, 300)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

# checkpoint_path = SIAMNET_PRE
checkpoint_path = SIAMNET_SFT

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to('cuda')
model.eval()
print('ready')


def sim_fn_siamnet(img1, img2):
    image_tensor_1 = preprocess(img1).unsqueeze(0)
    image_tensor_2 = preprocess(img2).unsqueeze(0)

    image_tensor_1 = image_tensor_1.to(next(model.parameters()).device)
    image_tensor_2 = image_tensor_2.to(next(model.parameters()).device)
    
    similarity_score = model.predict_similarity(image_tensor_1, image_tensor_2).item()
    
    del image_tensor_1
    del image_tensor_2

    return similarity_score

In [ ]:
evaluator = DomainFprEvaluator(
    data_dir=DOMAINNET_ROOT,
    sim_fn=sim_fn_siamnet,
    preprocess=None, # нужен если не вшит в sim_fn
    seed=2026
)

results = evaluator.evaluate(n_samples=5000, threshold=0.5, model_name = "Siamese Network (sft)", output_path = OUTPUT_DIR + "/fpr_domainnet.json")

In [ ]:
evaluator = DomainRecallEvaluator(
    data_dir=DOMAINNET_ROOT,
    sim_fn=sim_fn_siamnet,
    preprocess=None, # нужен если не вшит в sim_fn
    seed=2026
)

results = evaluator.evaluate(n_samples=1000, threshold=0.5, model_name = "Siamese Network (sft)", output_path = OUTPUT_DIR + "/recall_domainnet.json")

In [ ]:
from pathlib import Path
import random
import os
from PIL import Image
from IPython.display import display

# Параметры — должны совпадать с вашим evaluator'ом
data_dir = DOMAINNET_ROOT
val_size = 0.1
split_random_seed = 42

# Загрузка val-сплита (как в evaluator'е)
domain_to_val_paths = {}
rng = random.Random(split_random_seed)

for domain in sorted(os.listdir(data_dir)):
    domain_path = Path(data_dir) / domain
    if not domain_path.is_dir():
        continue
    all_paths = []
    for file in sorted(domain_path.iterdir()):
        if file.suffix.lower() in {".jpg", ".jpeg", ".png"}:
            all_paths.append(file)
    if not all_paths:
        continue
    shuffled = rng.sample(all_paths, len(all_paths))
    n_val = int(len(shuffled) * val_size)
    val_paths = shuffled[:n_val]
    if len(val_paths) >= 2:
        domain_to_val_paths[domain] = val_paths

# Генерация и отображение одной пары на домен
print("Примеры негативных пар (разные изображения из одного домена):\n")

for domain, paths in sorted(domain_to_val_paths.items()):
    p1, p2 = paths[0], paths[1]
    
    print(f"Domain: {domain}")
    print(f"  → {p1.name}")
    print(f"  → {p2.name}")
    
    # Загружаем и отображаем изображения рядом
    img1 = Image.open(p1).convert("RGB")
    img2 = Image.open(p2).convert("RGB")
    
    # Опционально: привести к одинаковой высоте для аккуратного отображения
    max_height = 200
    def resize_to_height(img, h):
        w = int(img.width * h / img.height)
        return img.resize((w, h), Image.LANCZOS)
    
    img1_r = resize_to_height(img1, max_height)
    img2_r = resize_to_height(img2, max_height)
    
    # Объединяем изображения горизонтально
    combined_width = img1_r.width + img2_r.width
    combined = Image.new('RGB', (combined_width, max_height))
    combined.paste(img1_r, (0, 0))
    combined.paste(img2_r, (img1_r.width, 0))
    
    display(combined)
    print("-" * 50)

### Negative Bench

#### Analysis of the test sample

In [ ]:
tokenizer = TransformTokenizer()
transformer = ImageTransformer()
batch_names = [
    "dataset_0", "dataset_1", "dataset_5", "dataset_9",
    "dataset_2", "dataset_18", "dataset_27", "dataset_36",
    "dataset_45", "dataset_270", "dataset_281", "dataset_150",
    "dataset_180"
]

negative_data = NegativeImagePairDataset(
        root_dir = NEGATIVE_ROOT,
        batch_names = batch_names,
        transformer = transformer,
        tokenizer = tokenizer,
        split = "val",
        val_size = 0.1,
        random_seed = 42,
        max_seq_len = 15,
        augmentation_p = 0.3
)

In [ ]:
len(negative_data._pairs_and_meta)

In [ ]:
category_to_pairs = {}
dataset_root = Path(NEGATIVE_ROOT)

In [ ]:
def load_pairs(json_path: str) -> None:
    with open(json_path, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    for entry in metadata:
        c1 = entry.get("image_1_class")
        c2 = entry.get("image_2_class")
        if not (c1 and c2 and c1 == c2):
            continue  # skip cross-domain or missing classes

        p1 = dataset_root / entry["batch_name"] / "dataset" / entry["image_1"]
        p2 = dataset_root / entry["batch_name"] / "dataset" / entry["image_2"]
        category_to_pairs.setdefault(c1, []).append((p1, p2))

In [ ]:
load_pairs(NEGATIVE_JSON)

In [ ]:
for cat in category_to_pairs.keys():
    print(cat, len(category_to_pairs[cat]))

In [ ]:
plot_classification_histogram("negative_test_dataset_meta.json")

In [ ]:
sampler = DatasetSampler(NEGATIVE_ROOT, NEGATIVE_JSON)

In [ ]:
# sampler.sample_and_display()

#### Evaluation of sequence prediction models (ViT based, EfficientNet based)

In [ ]:
preprocess = transforms.Compose([
            transforms.Resize((300, 300)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

config = OmegaConf.load(CONFIG_EFFNET)
model = ImageTransformPredictor(config.model)

# checkpoint_path = EFFNET_PRE
checkpoint_path = EFFNET_SFT

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to('cuda')
model.eval()
print('ready')

# model.load_state_dict(torch.load('checkpoints/itp_011/model_weights.pth'))
# model.to('cpu')
# model.eval()
# print('ready')

def sim_fn_itp_effnet(img1, img2):
    image_tensor_1 = preprocess(img1).unsqueeze(0).to('cuda')
    image_tensor_2 = preprocess(img2).unsqueeze(0).to('cuda')

    generated = model.generate(image_tensor_1, image_tensor_2, max_new_tokens=10, do_sample=False)
    if generated[0][1] == 2: # выдало пустую последовательность
        similarity_score = 0.
    else:
        similarity_score = 1.
    del image_tensor_1, image_tensor_2, generated
    return similarity_score

In [ ]:
preprocess = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

config = OmegaConf.load(CONFIG_VIT)
model = ImageTransformPredictor(config.model)

checkpoint_path = VIT_PRE
# checkpoint_path = VIT_SFT

checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to('cpu')
model.eval()
print('ready')

def sim_fn_itp_vit(img1, img2):
    image_tensor_1 = preprocess(img1).unsqueeze(0)
    image_tensor_2 = preprocess(img2).unsqueeze(0)

    generated = model.generate(image_tensor_1, image_tensor_2, max_new_tokens=10, do_sample=False)
    if generated[0][1] == 2: # выдало пустую последовательность
        similarity_score = 0.
    else:
        similarity_score = 1.
    del image_tensor_1, image_tensor_2, generated
    return similarity_score

In [ ]:
evaluator = RecallRobustnessEvaluator(
    dataset_root=NEGATIVE_ROOT,
    json_path="negative_test_dataset_meta.json",
    sim_fn=sim_fn_itp_effnet,
    preprocess=None, # нужен если не вшит в sim_fn
    seed=2025
)

results = evaluator.evaluate(n_samples=1000, threshold=0.5, model_name = "ours (EfficientNet-B3) before sft", output_path = OUTPUT_DIR + "/recall_negative_dataset.json")

In [ ]:
evaluator = RecallRobustnessEvaluator(
    dataset_root=NEGATIVE_ROOT,
    json_path="negative_test_dataset_meta.json",
    sim_fn=sim_fn_itp_effnet,
    preprocess=None, # нужен если не вшит в sim_fn
    seed=2025
)

results = evaluator.evaluate(n_samples=1000, threshold=0.5, model_name = "ours (EfficientNet-B3)", output_path = OUTPUT_DIR + "/recall_negative_dataset.json")

In [ ]:
evaluator = FprEvaluator(
    dataset_root=NEGATIVE_ROOT,
    json_path="negative_test_dataset_meta.json",
    sim_fn=sim_fn_itp_effnet,
    preprocess=None, # нужен если не вшит в sim_fn
    seed=2025
)

results = evaluator.evaluate(n_samples=1000, threshold=0.5, model_name = "ours (EfficientNet-B3) before sft", output_path = OUTPUT_DIR + "/fpr_negative_dataset.json")

In [ ]:
evaluator = FprEvaluator(
    dataset_root=NEGATIVE_ROOT,
    json_path="negative_test_dataset_meta.json",
    sim_fn=sim_fn_itp_vit,
    preprocess=None, # нужен если не вшит в sim_fn
    seed=2025
)

results = evaluator.evaluate(n_samples=1000, threshold=0.5, model_name = "ours (ViT-B/16) before sft", output_path = OUTPUT_DIR + "/fpr_negative_dataset.json")

#### Comparison with Qwen3-VL-4B-Instruct in the task of predicting the transformation sequence

In [ ]:
import torch

from transformers import AutoTokenizer, AutoProcessor
from transformers import Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from PIL import Image

if not hasattr(torch.compiler, "is_compiling"):
    torch.compiler.is_compiling = lambda: False

In [ ]:
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct",
)

# Store processor for preprocessing
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")

model.to("cuda")
print("ready")

In [ ]:
prompt_class_prediction = """You are an expert in image forensics. Determine whether Image B is a plagiarized version of Image A, meaning it was obtained by applying only the following allowed transformations to Image A:  
- Geometric: rotate_90, rotate_180, rotate_270, horizontal_flip, vertical_flip  
- Photometric: grayscale, color_jitter  
- Structural: crop  
- Noise/compression: noise_adding, jpeg_artefacts  

If Image B can be produced from Image A using any combination of these operations (in any order, without repetition), output 1.  
If Image B involves any other transformation (e.g., blur, resize, perspective warp, object removal, or unrelated content), output 0.

Output only a single digit: 1 or 0. Do not add any other text, explanation, or punctuation."""

In [ ]:
image = Image.open('../assets/dog.jpg')
transformed_image, sequence = transformer.transform_by_length(image, 2)

print(sequence)
transformed_image.show()

In [ ]:
import re
import torch

def sim_fn_qwen(img1, img2):
    # Ensure model is in eval mode to disable dropout, etc.
    was_training = model.training
    model.eval()
    
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": img1},
            {"type": "image", "image": img2},
            {"type": "text", "text": prompt_class_prediction},
        ],
    }]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():  # critical: disable gradient tracking
        generated_ids = model.generate(**inputs, max_new_tokens=128)

    # Extract only the needed part
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0].strip()

    # Clean up GPU tensors immediately
    del inputs, generated_ids, generated_ids_trimmed
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # optional, but safe for long runs

    match = re.fullmatch(r'[01]', output_text)
    if match:
        return float(match.group(0))
    else:
        raise ValueError(f"Unexpected model output: '{output_text}'. Expected '0' or '1'.")

In [ ]:
sim_fn_qwen(image, transformed_image)

In [ ]:
evaluator = RecallRobustnessEvaluator(
    dataset_root=NEGATIVE_ROOT,
    json_path="negative_test_dataset_meta.json",
    sim_fn=sim_fn_qwen,
    preprocess=None, # нужен если не вшит в sim_fn
    seed=2025
)

results = evaluator.evaluate(n_samples=1000, threshold=0.5, model_name = "Qwen3-VL-4B", output_path = OUTPUT_DIR + "/recall_negative_dataset.json")

In [ ]:
evaluator = FprEvaluator(
    dataset_root=NEGATIVE_ROOT,
    json_path="negative_test_dataset_meta.json",
    sim_fn=sim_fn_qwen,
    preprocess=None, # нужен если не вшит в sim_fn
    seed=2025
)

results = evaluator.evaluate(n_samples=1000, threshold=0.5, model_name = "Qwen3-VL-4B", output_path = OUTPUT_DIR + "/fpr_negative_dataset.json")

#### SiamNet

In [ ]:
evaluator = FprEvaluator(
    dataset_root=NEGATIVE_ROOT,
    json_path=NEGATIVE_JSON,
    sim_fn=sim_fn_siamnet,
    preprocess=None, # нужен если не вшит в sim_fn
    seed=2026
)

results = evaluator.evaluate(n_samples=1000, threshold=0.5, model_name = "Siamese Network (sft)", output_path = OUTPUT_DIR + "/fpr_negative_dataset.json")

In [ ]:
evaluator = RecallRobustnessEvaluator(
    dataset_root=NEGATIVE_ROOT,
    json_path=NEGATIVE_JSON,
    sim_fn=sim_fn_siamnet,
    preprocess=None, # нужен если не вшит в sim_fn
    seed=2026
)

results = evaluator.evaluate(n_samples=1000, threshold=0.5, model_name = "Siamese Network (sft)", output_path = OUTPUT_DIR + "/recall_negative_dataset.json")
# results = evaluator.evaluate(n_samples=1000, threshold=0.5, model_name = "Siamese Network", output_path = "benchmarking/metrics/metric_siamnet_new/recall_negative_dataset.json")